# Подготовка дополнительных отзывов для независимой разметки

Источник: [Women's E-Commerce Clothing Reviews](https://www.kaggle.com/datasets/nicapotato/womens-ecommerce-clothing-reviews).
На странице публикации указана лицензия CC0: Public Domain.
Корпус выбран из альтернатив, перечисленных в codebook.

Отбираются 120 полных англоязычных отзывов длиной 5–80 слов для основной разметки. Предварительный отбор по
рейтингам 1–2 и 4–5 используется для формирования пула, но не определяет итоговые
человеческие метки. Рейтинги и рекомендации не включаются в файлы разметки.

Исходный UCI используется только для проверки пересечений. Источники, параметры
отбора и лицензия фиксируются отдельно. Семантические дубликаты данной проверкой
полностью не исключаются. Добавление отзывов об одежде изменяет тематический состав набора.


In [ ]:
from pathlib import Path
import os
import io
import zipfile
import requests
try:
    from google.colab import drive
except ImportError:
    STORAGE = Path(os.environ.get('COURSEWORK_DATA_ROOT', '.local-data')).resolve()
else:
    drive.mount('/content/drive')
    STORAGE = Path('/content/drive/MyDrive/data-enrichment-coursework')
original = STORAGE / 'processed/text/reviews_original.csv'
if not original.exists():
    raise FileNotFoundError('Не найдена исходная таблица UCI: ' + str(original))
RAW = STORAGE / 'raw/manual_reviews'
RAW.mkdir(parents=True, exist_ok=True)


## 1. Загрузка внешнего источника


In [ ]:
URL = 'https://www.kaggle.com/api/v1/datasets/download/nicapotato/womens-ecommerce-clothing-reviews'
archive = RAW / 'clothing_reviews.zip'
source = RAW / 'Womens Clothing E-Commerce Reviews.csv'
if not archive.exists():
    response = requests.get(URL, timeout=120)
    response.raise_for_status()
    # Проверка архива до сохранения исключает HTML-страницу авторизации.
    with zipfile.ZipFile(io.BytesIO(response.content)) as bundle:
        content = bundle.read('Womens Clothing E-Commerce Reviews.csv')
    archive.write_bytes(response.content)
    source.write_bytes(content)
else:
    with zipfile.ZipFile(archive) as bundle:
        source.write_bytes(bundle.read('Womens Clothing E-Commerce Reviews.csv'))


## 2. Проверка пересечений и формирование набора


In [ ]:
"""Prepare blind annotation batches from the licensed external review corpus."""
import hashlib
import json
import os
import re
from pathlib import Path
import pandas as pd

ROOT = STORAGE

RAW = STORAGE / 'raw/manual_reviews'
OUT = STORAGE / 'manual/text'
OUT.mkdir(parents=True, exist_ok=True)
source = RAW / 'Womens Clothing E-Commerce Reviews.csv'
df = pd.read_csv(source)
source_rows = len(df)
df['source_row'] = df.index
df = df.dropna(subset=['Review Text']).copy()
def norm(s):
    return ' '.join(re.findall(r'\w+', str(s).casefold()))
df['normalized'] = df['Review Text'].map(norm)
df['words'] = df['Review Text'].str.split().str.len()
df = df[df.words.between(5,80)].drop_duplicates('normalized')
original = pd.read_csv(STORAGE / 'processed/text/reviews_original.csv')
originals = [norm(t) for t in original.text]
df = df[~df.normalized.isin(set(originals))]
# Rating is used only for balanced candidate selection, never as an annotation.
buckets = [df[df.Rating.isin([1,2])], df[df.Rating.isin([4,5])]]
selected = []
for bucket in buckets:
    accepted = []
    for _, row in bucket.sample(frac=1, random_state=42).iterrows():
        text = row.normalized
        if any(text in other or other in text for other in originals if other):
            continue
        accepted.append(row)
        if len(accepted) == 80:
            break
    assert len(accepted) == 80
    selected.append(pd.DataFrame(accepted))

metadata = []
for name, start, stop in [('main',10,70)]:
    batch = pd.concat([b.iloc[start:stop] for b in selected]).sample(frac=1,random_state=73)
    records = []
    for _, row in batch.iterrows():
        identifier = 'review_' + hashlib.sha256(f"clothing:{row.source_row}".encode()).hexdigest()[:12]
        records.append({'data': {'id':identifier,'text':row['Review Text']}})
        metadata.append({'id':identifier,'batch':name,'source_row':int(row.source_row),
                         'source':'Womens E-Commerce Clothing Reviews',
                         'url':'https://www.kaggle.com/datasets/nicapotato/womens-ecommerce-clothing-reviews',
                         'license_on_source_page':'CC0: Public Domain','access_date':'2026-11-08'})
    (OUT / f'{name}.json').write_text(json.dumps(records,ensure_ascii=False,indent=2),encoding='utf-8')
    pd.DataFrame([r['data'] for r in records]).to_csv(OUT / f'{name}.csv',index=False)
pd.DataFrame(metadata).to_csv(OUT/'sources.csv',index=False)
summary={'source_rows':source_rows,'main':120,'seed':42,
         'archive_sha256':hashlib.sha256((RAW/'clothing_reviews.zip').read_bytes()).hexdigest(),
         'checks':['5–80 words','nonempty text','normalized duplicate removal',
                   'no normalized exact matches or substrings with original UCI'],
         'limitations':'No semantic duplicate detection; final labels require two independent human annotations.'}
(OUT/'selection_summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding='utf-8')
print(json.dumps(summary,ensure_ascii=False,indent=2))


## 3. Документирование источника и интерфейс разметки


In [ ]:
(OUT / 'label_studio.xml').write_text('<View>\n  <Text name="review" value="$text"/>\n  <Choices name="sentiment" toName="review" choice="single" required="true">\n    <Choice value="positive"/>\n    <Choice value="negative"/>\n    <Choice value="disputed"/>\n  </Choices>\n  <Header value="Время разметки в секундах (без перерывов)"/>\n  <TextArea name="time_seconds" toName="review" required="true" rows="1"/>\n  <Header value="Комментарий и причина сомнения"/>\n  <TextArea name="comment" toName="review" rows="3"/>\n</View>\n', encoding='utf-8')
(OUT / 'README.md').write_text("# Дополнительные отзывы: независимая разметка\n\nИсточник: [Women's E-Commerce Clothing Reviews](https://www.kaggle.com/datasets/nicapotato/womens-ecommerce-clothing-reviews), публикация nicapotato на Kaggle. На странице указана лицензия [CC0: Public Domain](https://creativecommons.org/publicdomain/zero/1.0/). Выбран предусмотренный codebook альтернативный источник. Данные относятся к обезличенному коммерческому источнику; конкретный магазин автором не раскрывается.\n\n| Файл | Назначение | Объём |\n| --- | --- | --- |\n| main.json | Основная независимая разметка | 120 |\n\nCSV содержат те же тексты. Рейтинги и исходные рекомендации исключены из файлов разметки. Классы не зашифрованы в идентификаторах или порядке строк. Источники записаны в sources.csv. Предварительный отбор по рейтингам не гарантирует баланс человеческих меток.\n\n## Протокол\n\n1. Создать два отдельных проекта Label Studio: A1 и A2. Установить интерфейс из label_studio.xml и импортировать в каждый один и тот же main.json.\n2. Каждый участник независимо размечает все 120 текстов по codebook, записывает время в секундах и комментарии. Метки: positive = 1, negative = 0, disputed = спорный. Ответы другого участника не просматриваются.\n3. Сохранить оба экспорта до обсуждения. Согласованность вычисляется по независимым ответам, итоговые метки хранятся отдельно. Если после исключений осталось менее 50 примеров какого-либо класса, потребуется дополнительный сбор; метки ради баланса не изменяются.\n\n## Ограничения\n\nОтобраны полные тексты длиной 5–80 слов. Исключены нормализованные повторы, совпадения и вхождения текстов исходного UCI. Проверка не исключает смысловые перефразирования. Тексты сохранены без перевода и генерации. Корпус относится к одежде: это изменение тематики относительно товаров, ресторанов и фильмов UCI; предварительный отбор по крайним рейтингам также влияет на состав выборки.\n", encoding='utf-8')


In [ ]:
source_record = {
    'name': "Women's E-Commerce Clothing Reviews",
    'url': 'https://www.kaggle.com/datasets/nicapotato/womens-ecommerce-clothing-reviews',
    'publisher': 'nicapotato (Kaggle)',
    'license_on_publisher_page': 'CC0: Public Domain',
    'license_url': 'https://creativecommons.org/publicdomain/zero/1.0/',
    'source_identity': 'Anonymous but real commercial source, according to publisher',
    'scope': 'Additional real product reviews; no translation or generated text',
    'archive_sha256': summary['archive_sha256'],
    'selection_seed': 42,
}
(OUT/'source_license.json').write_text(json.dumps(source_record,ensure_ascii=False,indent=2),encoding='utf-8')
tasks=[]
for name, expected in [('main',120)]:
    rows=json.loads((OUT/f'{name}.json').read_text(encoding='utf-8'))
    assert len(rows)==expected
    assert all(set(r)=={'data'} and set(r['data'])=={'id','text'} for r in rows)
    tasks.extend(rows)
assert len({r['data']['id'] for r in tasks}) == 120
assert len({norm(r['data']['text']) for r in tasks}) == 120
print('Основной набор для независимой разметки: 120 отзывов.')
print('Каталог файлов разметки:', OUT)
